# 민영이랑 이야기해 볼 것
### XGBRegressor로 사고수 예측한 게 애매하대
- 사고수는 대부분 0이라 회귀보다 사고 발생 여부 = 사고수 > 0 분류가 더 적합함

### train/test 분리 없이 전체 데이터로 학습함
- 전체 데이터로 학습하고 바로 중요도 뽑으면 검증력이 약함
- train/test 분리해서 하기!
- 7:3, 8:2 정도로(학교에서 이렇게 배웠는데...!)

### MAE가 좋아 보여도 의미가 약하대
- 사고가 거의 0이면 전부 0으로 예측해도 MAE가 낮게 나올 수 있음

### 환승지수가 너무 크게 나왔대
- 그래서 0.35로 제한했는데 이건 약간 임의적?
- 설명은 가능하지만 더 설득력 있게 바꾸면 좋을듯!!

### 시간 기준 분리도 고려
- 랜덤 train/test분리도 가능하지만 그거보다
- 사고 예측 모델이면 과거 데이터로 학습하고 미래데이터를 예측하는 방식이 더 자연스럽대
- 근데 미래 데이터가 있나...?
- 이거 지피티한테 더 자세히 물어봐야겠다

### 클래스 불균형 처리 필요
- 사고발생=1인 데이터가 적기 때문에 scale_pos_weight같은 옵션을 줘야할 듯
- 아니면 모델이 대부분 0으로만 예측할 수 있으니까!

### +)
- Risk = w1*혼잡지수 + ... 로 직접 점수를 만드는 방식보다,
- XGBClassifier가 예측한 사고발생확률을 Risk로 사용하고,
- SHAP/변수중요도로 각 위험요인의 기여도를 해석하는 방식이 더 설득력있을 수도 있다고 해서 이거도 고려해보기!!

---

# <바꿔서 해볼 것>
### 메인 모델: XGBClassifier
- 우리 데이터는사고수가 대부분 0이라서 사고 건수를 맞히는 회귀보다
- 사고가 발생할 가능성이 있는지/없는지를 예측하는 분류 문제가 더 적합
- 사고수 > 0을 사고발생 여부로 바꾸고 XGBClassifier의 예측확률을 위험도 Risk로 사용해봐야겠음

### 평가지표:
- ROC-AUC: 전체적인 분류 성능을 봄
- PR-AUC: 사고처럼 드문 사건에서 모델이 위험 사례를 얼마나 잘 찾는지 보기 좋음
- Recall
- F1-score: 실제 사고 발생 지점을 놓치지 않는지 확인하는 지표
- Top 10% Recall: 위험도 상위 역들이 실제 사고역을 얼마나 포함하는지 보여줘서 정책 우선순위 선정에 적합
- 특히 공모전에서는 Top 위험역이 실제 사고역을 얼마나 잘 포함하는지가 중요해서 Top-K Recall을 넣는게 좋대!

### Risk = w1*혼잡지수 + w2*유입지수 + w3*구조지수 + w4*환승지수
- 여기서 가중치를 XGBRegressor feature importance에서 뽑기보다는 XGBClassifier의 예측확률 = Risk로 함 바꿔서 써볼게 이게 더 좋나!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df_final_all = pd.read_csv('/content/drive/MyDrive/서울시 빅데이터 공모전/data/processed/df_final_all_preprocessed.csv', encoding='utf-8')

/tmp/ipykernel_3935/1888928975.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final_all = pd.read_csv('/content/drive/MyDrive/서울시 빅데이터 공모전/data/processed/df_final_all_preprocessed.csv', encoding='utf-8')


In [ ]:
df_final_all.head(20)

,사용월,호선명,time_group,혼잡,역명,구조평균,구조최소,구조최대,환승,평일(일평균),토요일,일요일,사고수,유입,child_pop_sum,youth_pop_sum,middle_pop_sum,elder_pop_sum
0,202101,1호선,기타,85553,동대문,12.66250,10.0,19.0,1,54161.0,52147.0,39298.0,0,109.158033,2.371445,28.573098,38.102411,40.111080
1,202101,1호선,낮시간,227205,동대문,12.66250,10.0,19.0,1,54161.0,52147.0,39298.0,0,89.818268,4.140349,13.174589,27.672827,44.830502
2,202101,1호선,출근시간,50184,동대문,12.66250,10.0,19.0,1,54161.0,52147.0,39298.0,0,57.970107,1.030731,8.346038,19.046191,29.547147
3,202101,1호선,퇴근시간,132625,동대문,12.66250,10.0,19.0,1,54161.0,52147.0,39298.0,0,95.624161,3.856447,21.486049,28.811206,41.470459
4,202101,1호선,기타,34309,동묘앞,6.83750,3.0,12.0,1,51161.0,47054.0,36945.0,0,70.195318,2.336531,25.357979,22.212263,20.288545
5,202101,1호선,낮시간,255758,동묘앞,6.83750,3.0,12.0,1,51161.0,47054.0,36945.0,0,59.571030,2.421803,11.906279,18.392318,26.850630
6,202101,1호선,출근시간,31855,동묘앞,6.83750,3.0,12.0,1,51161.0,47054.0,36945.0,0,37.758642,0.627182,8.560263,14.044943,14.526254
7,202101,1호선,퇴근시간,111594,동묘앞,6.83750,3.0,12.0,1,51161.0,47054.0,36945.0,0,63.048366,3.533937,19.791535,19.631047,20.091848
8,202101,1호선,기타,232119,서울,10.82125,1.5,21.0,1,163998.0,156550.0,117615.0,0,34.795002,2.768720,13.940185,10.029385,8.056711
9,202101,1호선,낮시간,602536,서울,10.82125,1.5,21.0,1,163998.0,156550.0,117615.0,0,29.593260,1.310507,8.826405,10.836272,8.620076


In [ ]:
print(df_final_all.isna().sum())

사용월               0
호선명               0
time_group        0
혼잡                0
역명                0
구조평균              0
구조최소              0
구조최대              0
환승                0
평일(일평균)           0
토요일               0
일요일               0
사고수               0
유입                0
child_pop_sum     0
youth_pop_sum     0
middle_pop_sum    0
elder_pop_sum     0
dtype: int64


In [ ]:
df_final_all.duplicated(
    subset=['사용월','호선명', '역명', 'time_group']
).sum()

np.int64(0)

In [ ]:
import pandas as pd

# 숫자로 바꾸기
num_cols = ['유입', '구조최대']

for col in num_cols:
    df_final_all[col] = (
        df_final_all[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_final_all[col] = pd.to_numeric(df_final_all[col], errors='coerce')

# 혹시 변환 실패한 값 확인
print(df_final_all[num_cols].isna().sum())

유입      0
구조최대    0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# 숫자로 변환 (실패는 NaN)
df_final_all['혼잡'] = pd.to_numeric(df_final_all['혼잡'], errors='coerce')

# 이상값 제거 (현실적인 범위)
df_final_all.loc[df_final_all['혼잡'] > 10_000_000, '혼잡'] = np.nan

# NaN 처리 (중요)
df_final_all['혼잡'] = df_final_all['혼잡'].fillna(df_final_all['혼잡'].median())

In [ ]:
df_final_all['혼잡'].describe()

,혼잡
count,7.396400e+04
mean,2.226638e+05
std,2.135536e+05
min,0.000000e+00
25%,9.055975e+04
50%,1.603690e+05
75%,2.778758e+05
max,3.089838e+06


In [ ]:
df_final_all.sort_values('혼잡', ascending=False)[
    ['사용월', '역명', '호선명', 'time_group', '혼잡']
].head(20)

,사용월,역명,호선명,time_group,혼잡
38253,202309,잠실,2호선,낮시간,3089838.0
38085,202309,강남,2호선,낮시간,2992000.0
38281,202309,홍대입구,2호선,낮시간,2878930.0
38255,202309,잠실,2호선,퇴근시간,2787734.0
38283,202309,홍대입구,2호선,퇴근시간,2758634.0
38087,202309,강남,2호선,퇴근시간,2744266.0
38297,202309,고속터미널,3호선,낮시간,2505324.0
38053,202309,서울,1호선,낮시간,2295794.0
38149,202309,삼성,2호선,낮시간,2103186.0
70549,202512,잠실,2호선,낮시간,2060059.0


In [ ]:
outlier_congestion = df_final_all[df_final_all['혼잡'] > 10_000_000]

outlier_congestion[
    ['사용월', '역명', '호선명', 'time_group', '혼잡']
].head(50)

,사용월,역명,호선명,time_group,혼잡


In [ ]:
df_final_all['혼잡지수'] = df_final_all['혼잡'] / df_final_all['혼잡'].max()
df_final_all['혼잡지수'].describe()

,혼잡지수
count,73964.000000
mean,0.072063
std,0.069115
min,0.000000
25%,0.029309
50%,0.051902
75%,0.089932
max,1.000000


In [ ]:
df_check = df_final_all[['역명','호선명','time_group','사고수']].drop_duplicates()

df_check.groupby(['역명','호선명','time_group'])['사고수'].first()

역명     호선명  time_group
가락시장   3호선  기타            0
            낮시간           0
            출근시간          0
            퇴근시간          0
       8호선  기타            0
                         ..
효창공원앞  6호선  퇴근시간          0
흑석     9호선  기타            0
            낮시간           0
            출근시간          0
            퇴근시간          0
Name: 사고수, Length: 1200, dtype: int64

In [ ]:
df_final_all.groupby(['역명','호선명','time_group'])['사고수'].first()

역명     호선명  time_group
가락시장   3호선  기타            0
            낮시간           0
            출근시간          0
            퇴근시간          0
       8호선  기타            0
                         ..
효창공원앞  6호선  퇴근시간          0
흑석     9호선  기타            0
            낮시간           0
            출근시간          0
            퇴근시간          0
Name: 사고수, Length: 1200, dtype: int64

In [ ]:
print(df_final_all.columns.tolist())

['사용월', '호선명', 'time_group', '혼잡', '역명', '구조평균', '구조최소', '구조최대', '환승', '평일(일평균)', '토요일', '일요일', '사고수', '유입', 'child_pop_sum', 'youth_pop_sum', 'middle_pop_sum', 'elder_pop_sum', '혼잡지수']


In [ ]:
df_final_all['혼잡지수'] = df_final_all['혼잡'] / df_final_all['혼잡'].max()
df_final_all['유입지수'] = df_final_all['유입'] / df_final_all['유입'].max()
df_final_all['구조지수'] = df_final_all['구조최대'] / df_final_all['구조최대'].max()

In [ ]:
df_final_all['환승량'] = df_final_all[['평일(일평균)', '토요일', '일요일']].mean(axis=1)

In [ ]:
df_final_all['환승지수'] = df_final_all['환승량'] / df_final_all['환승량'].max()

In [ ]:
print(df_final_all.isna().sum())

사용월               0
호선명               0
time_group        0
혼잡                0
역명                0
구조평균              0
구조최소              0
구조최대              0
환승                0
평일(일평균)           0
토요일               0
일요일               0
사고수               0
유입                0
child_pop_sum     0
youth_pop_sum     0
middle_pop_sum    0
elder_pop_sum     0
혼잡지수              0
유입지수              0
구조지수              0
환승량               0
환승지수              0
dtype: int64


In [ ]:
df_final_all.head(50)

,사용월,호선명,time_group,혼잡,역명,구조평균,구조최소,구조최대,환승,평일(일평균),...,유입,child_pop_sum,youth_pop_sum,middle_pop_sum,elder_pop_sum,혼잡지수,유입지수,구조지수,환승량,환승지수
0,202101,1호선,기타,85553.0,동대문,12.66250,10.0,19.0,1,54161.0,...,109.158033,2.371445,28.573098,38.102411,40.111080,0.027689,0.425565,0.678571,48535.333333,0.219240
1,202101,1호선,낮시간,227205.0,동대문,12.66250,10.0,19.0,1,54161.0,...,89.818268,4.140349,13.174589,27.672827,44.830502,0.073533,0.350167,0.678571,48535.333333,0.219240
2,202101,1호선,출근시간,50184.0,동대문,12.66250,10.0,19.0,1,54161.0,...,57.970107,1.030731,8.346038,19.046191,29.547147,0.016242,0.226003,0.678571,48535.333333,0.219240
3,202101,1호선,퇴근시간,132625.0,동대문,12.66250,10.0,19.0,1,54161.0,...,95.624161,3.856447,21.486049,28.811206,41.470459,0.042923,0.372802,0.678571,48535.333333,0.219240
4,202101,1호선,기타,34309.0,동묘앞,6.83750,3.0,12.0,1,51161.0,...,70.195318,2.336531,25.357979,22.212263,20.288545,0.011104,0.273665,0.428571,45053.333333,0.203512
5,202101,1호선,낮시간,255758.0,동묘앞,6.83750,3.0,12.0,1,51161.0,...,59.571030,2.421803,11.906279,18.392318,26.850630,0.082774,0.232245,0.428571,45053.333333,0.203512
6,202101,1호선,출근시간,31855.0,동묘앞,6.83750,3.0,12.0,1,51161.0,...,37.758642,0.627182,8.560263,14.044943,14.526254,0.010310,0.147206,0.428571,45053.333333,0.203512
7,202101,1호선,퇴근시간,111594.0,동묘앞,6.83750,3.0,12.0,1,51161.0,...,63.048366,3.533937,19.791535,19.631047,20.091848,0.036116,0.245801,0.428571,45053.333333,0.203512
8,202101,1호선,기타,232119.0,서울,10.82125,1.5,21.0,1,163998.0,...,34.795002,2.768720,13.940185,10.029385,8.056711,0.075123,0.135652,0.750000,146054.333333,0.659746
9,202101,1호선,낮시간,602536.0,서울,10.82125,1.5,21.0,1,163998.0,...,29.593260,1.310507,8.826405,10.836272,8.620076,0.195006,0.115373,0.750000,146054.333333,0.659746


**XGBoost**

In [ ]:
from xgboost import XGBRegressor
import pandas as pd

In [ ]:
X = df_final_all[['혼잡지수', '유입지수', '구조지수', '환승지수']]
y = df_final_all['사고수']

In [ ]:
model = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

환승지수    0.462503
혼잡지수    0.202690
구조지수    0.179350
유입지수    0.155457
dtype: float32


In [ ]:
# 하이퍼파라미터 최적화

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error
import numpy as np
import pandas as pd

X = df_final_all[['혼잡지수', '유입지수', '구조지수', '환승지수']]
y = df_final_all['사고수']

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 10],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [1, 3, 5, 10]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X, y)

print("Best params:")
print(search.best_params_)

print("Best score:", search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params:
{'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best score: -0.022365663200616837


In [ ]:
best_model = search.best_estimator_

importance = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance

,0
환승지수,0.540580
혼잡지수,0.164838
구조지수,0.158442
유입지수,0.136140


In [ ]:
importance_norm = importance / importance.sum()
importance_norm

,0
환승지수,0.540580
혼잡지수,0.164838
구조지수,0.158442
유입지수,0.136140


In [ ]:
weights = importance_norm.copy()

# 환승지수 가중치 최대 0.35로 제한
if weights['환승지수'] > 0.35:
    extra = weights['환승지수'] - 0.35
    weights['환승지수'] = 0.35

    other_cols = ['혼잡지수', '유입지수', '구조지수']
    weights[other_cols] = weights[other_cols] + extra * (weights[other_cols] / weights[other_cols].sum())

weights

,0
환승지수,0.350000
혼잡지수,0.233218
구조지수,0.224168
유입지수,0.192614


In [ ]:
df_final_all['Risk'] = (
    weights['혼잡지수'] * df_final_all['혼잡지수'] +
    weights['유입지수'] * df_final_all['유입지수'] +
    weights['구조지수'] * df_final_all['구조지수'] +
    weights['환승지수'] * df_final_all['환승지수']
)

In [ ]:
df_final_all['Risk'].head()

,Risk
0,0.317276
1,0.313445
2,0.276167
3,0.310666
4,0.222602


In [ ]:
top_risk_station = (
    df_final_all
    .groupby(['역명', '호선명', 'time_group'], as_index=False)
    .agg(
        Risk=('Risk', 'mean'),
        혼잡지수=('혼잡지수', 'mean'),
        유입지수=('유입지수', 'mean'),
        구조지수=('구조지수', 'mean'),
        환승지수=('환승지수', 'mean'),
        사고수=('사고수', 'max')
    )
    .sort_values('Risk', ascending=False)
    .head(20)
)

top_risk_station

,역명,호선명,time_group,Risk,혼잡지수,유입지수,구조지수,환승지수,사고수
332,동대문역사문화공원,5호선,기타,0.530880,0.007414,0.538952,0.571429,0.849273,0
335,동대문역사문화공원,5호선,퇴근시간,0.526513,0.018860,0.502418,0.571429,0.849273,0
703,신도림,2호선,퇴근시간,0.524559,0.279230,0.256430,0.267857,1.000000,4
701,신도림,2호선,낮시간,0.520684,0.306471,0.203331,0.267857,1.000000,5
333,동대문역사문화공원,5호선,낮시간,0.520639,0.016913,0.474278,0.571429,0.849273,0
329,동대문역사문화공원,4호선,낮시간,0.513233,0.114660,0.109652,0.750000,0.849273,0
331,동대문역사문화공원,4호선,퇴근시간,0.510579,0.103024,0.109965,0.750000,0.849273,0
585,서울,1호선,낮시간,0.510135,0.381085,0.115373,0.750000,0.659746,2
328,동대문역사문화공원,4호선,기타,0.502951,0.058771,0.123940,0.750000,0.849273,0
587,서울,1호선,퇴근시간,0.496567,0.313559,0.126693,0.750000,0.659746,1
